<a href="https://colab.research.google.com/github/Sanjay-Vasudhevan/AI-Document-Workflow-Automation/blob/main/Predictive_Maintenance_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [6]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report,confusion_matrix,precision_recall_curve,auc,roc_auc_score,f1_score,precision_score,recall_score)

In [7]:
dataset_url = "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
zip_path = "ai4i2020.zip"
extract_dir = "ai4i_data"

In [8]:
urllib.request.urlretrieve(dataset_url, zip_path)

('ai4i2020.zip', <http.client.HTTPMessage at 0x7ce5182fd480>)

In [9]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [10]:
csv_filename = [
    f for f in os.listdir(extract_dir)
    if f.endswith('.csv')
][0]

In [11]:
df = pd.read_csv(os.path.join(extract_dir, csv_filename))

In [12]:
print(f"Dataset Loaded Successfully! Shape: {df.shape}")

print(df.head())

Dataset Loaded Successfully! Shape: (10000, 14)
   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0  

In [13]:
df.columns = [c.strip() for c in df.columns]

In [14]:
df['Temp_Diff'] = (
    df['Process temperature [K]']
    - df['Air temperature [K]']
)

In [15]:
df['Power_Proxy'] = (
    df['Rotational speed [rpm]']
    * df['Torque [Nm]']
)

In [16]:
target_col = 'Machine failure'

In [17]:
leakage_cols = [
    'TWF', 'HDF', 'PWF', 'OSF', 'RNF',
    'UDI', 'Product ID'
]

In [18]:
feature_cols = [
    c for c in df.columns
    if c not in leakage_cols + [target_col]
]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
num_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Temp_Diff',
    'Power_Proxy'
]

cat_features = ['Type']

In [21]:
preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_features),('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)])

In [22]:
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

In [23]:
results = {}

In [27]:
for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipe.fit(X_train, y_train)
    y_preds = pipe.predict(X_test)
    y_probs = pipe.predict_proba(X_test)[:, 1]

    precision, recall, _ = precision_recall_curve(y_test, y_probs)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(y_test, y_probs)

    results[name] = {
        'pipeline': pipe,
        'pr_auc': pr_auc,
        'roc_auc': roc_auc,
        'f1': f1_score(y_test, y_preds)
    }
    print(f"{name:20s} | PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc_auc:.4f} | F1: {results[name]['f1']:.4f}")



Logistic Regression  | PR-AUC: 0.4611 | ROC-AUC: 0.9340 | F1: 0.2943
Random Forest        | PR-AUC: 0.8525 | ROC-AUC: 0.9584 | F1: 0.7863
Gradient Boosting    | PR-AUC: 0.8995 | ROC-AUC: 0.9759 | F1: 0.8548


In [28]:
best_model_name = max(results, key=lambda k: results[k]['pr_auc'])
print(f"\nBest Performing Model: {best_model_name}")

best_pipeline = results[best_model_name]['pipeline']


Best Performing Model: Gradient Boosting


In [29]:
print("\n=== Step 4: Threshold Optimization ===")
y_probs_best = best_pipeline.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs_best)


f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

print(f"Optimal Classification Threshold for Max F1: {optimal_threshold:.4f}")

y_pred_optimal = (y_probs_best >= optimal_threshold).astype(int)
print("\nFinal Classification Report (At Optimal Threshold):")
print(classification_report(y_test, y_pred_optimal))


=== Step 4: Threshold Optimization ===
Optimal Classification Threshold for Max F1: 0.5931

Final Classification Report (At Optimal Threshold):
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1932
           1       1.00      0.78      0.88        68

    accuracy                           0.99      2000
   macro avg       1.00      0.89      0.94      2000
weighted avg       0.99      0.99      0.99      2000



In [31]:
print("\n=== Step 5: Exporting Serialized Model Artifacts ===")
os.makedirs("model_assets", exist_ok=True)
model_path = "model_assets/predictive_maintenance_pipeline.joblib"

joblib.dump({
    'pipeline': best_pipeline,
    'optimal_threshold': float(optimal_threshold),
    'num_features': num_features,
    'cat_features': cat_features
}, model_path)

print(f"Saved complete ML Pipeline artifact to: {model_path}")


=== Step 5: Exporting Serialized Model Artifacts ===
Saved complete ML Pipeline artifact to: model_assets/predictive_maintenance_pipeline.joblib


In [32]:
from google.colab import files

# Triggers an automatic download of the saved model pipeline
files.download('model_assets/predictive_maintenance_pipeline.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>